In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.gridspec as gridspec
import numpy as np
from scipy.ndimage import gaussian_filter1d
from scipy.signal import savgol_filter
import fsspec

# ── Load Data ──────────────────────────────────────────────────────────────────
def load_specific_parquet(url):
    with fsspec.open(url, 'rb') as f:
        return pd.read_parquet(f)

url  = "gs://agntworks-data-dev/wheelsup/processed/df_bookings_metros_final_merge.parquet"
metros_final_merge = load_specific_parquet(url)
Data = metros_final_merge.copy()

#---------------Loading support files------------------------------------------
all_DI_sheets = pd.read_excel('DI_Analysis_LJ_SMID.xlsx', sheet_name=None)
all_LI_sheets = pd.read_excel('LI_Analysis_LJ_SMID.xlsx', sheet_name=None)

#---------------Loading the toggle percent files--------------------------------
all_DI_LI_toggle = pd.read_excel("DI_LI_Analysis_Report with toggle % v2pr.xlsx",sheet_name=None)


In [10]:
#-------------Taking the toggle value for each flight--------------------
DI_Toggle = all_DI_LI_toggle['DI Light SMID']
LI_Toggle = all_DI_LI_toggle['LI Light SMID']

#--------------Adding AirCraft Type column-----------------------------------
all_DI_sheets['DI Light Jet']['AirCraftType'] = 'Light'
all_DI_sheets['DI SMID']['AirCraftType'] = 'Super Midsize'
all_LI_sheets['LI Light Jet']['AirCraftType'] = 'Light'
all_LI_sheets['LI SMID']['AirCraftType'] = 'Super Midsize'

#-----------------Merging both DI and LI data---------------------------------
DI_data = pd.concat([all_DI_sheets['DI Light Jet'], all_DI_sheets['DI SMID']])
LI_data = pd.concat([all_LI_sheets['LI Light Jet'], all_LI_sheets['LI SMID']])

#-------------Renaming the columns Name to match with mapping data------------
DI_data.rename(columns={'Route': 'corridor', 'day_name': 'DOW', 'hour_bucket': 'TOD'}, inplace=True)
LI_data.rename(columns={'Route': 'corridor', 'day_name': 'DOW', 'hour_bucket': 'TOD'}, inplace=True)

#------------Taking only important columns-------------------------------------
DI_data = DI_data[['AirCraftType', 'corridor', 'DOW', 'TOD', 'Tier_gap']]
LI_data = LI_data[['AirCraftType', 'corridor', 'DOW', 'TOD', 'WU_LI_gap']]

# ✅ FIX 1: Deduplicate DI and LI lookup tables BEFORE merging
# If duplicates exist, keep last (or use mean/first depending on your logic)
DI_data = DI_data.drop_duplicates(subset=['AirCraftType', 'corridor', 'DOW', 'TOD'], keep='last')
LI_data = LI_data.drop_duplicates(subset=['AirCraftType', 'corridor', 'DOW', 'TOD'], keep='last')

# TOD Mapping
DI_tod_mapping = {
    '07:00-10:00': '07:00-09:59', '10:00-13:00': '10:00-12:59',
    '13:00-16:00': '13:00-15:59', '16:00-19:00': '16:00-18:59',
    '19:00-24:00': '19:00-23:59', '00:00-07:00': '00:00-06:59'
}
LI_tod_mapping = {
    '07:00–10:00': '07:00-09:59', '10:00–13:00': '10:00-12:59',
    '13:00–16:00': '13:00-15:59', '16:00–19:00': '16:00-18:59',
    '19:00–24:00': '19:00-23:59', '00:00–07:00': '00:00-06:59'
}

DI_data['TOD'] = DI_data['TOD'].str.strip().str.replace('–', '-').map(DI_tod_mapping)
LI_data['TOD'] = LI_data['TOD'].map(LI_tod_mapping)

# ✅ FIX 2: Deduplicate Toggle tables too
DI_Toggle = DI_Toggle.drop_duplicates(subset=['Tier_gap', 'AirCraftType'], keep='last')
LI_Toggle = LI_Toggle.drop_duplicates(subset=['WU_LI_gap', 'AirCraftType'], keep='last')

#-------------Preparing main flight data---------------
data = Data.copy()
data['corridor'] = data['corridor'].str.replace(' - ', '->')
data.rename(columns={'flightactualAircraftCabinName': 'AirCraftType'}, inplace=True)
data['AirCraftType'] = data['AirCraftType'].map({
    'Super Midsize': 'Super Midsize', 'Premium Super-Mid': 'Super Midsize',
    'Premium Light': 'Light', 'Light': 'Light'
})

map_data = data[['AirCraftType', 'corridor', 'DOW', 'TOD']].copy()

# ✅ FIX 3: Merge DI and LI into a SINGLE lookup table first, then merge once with map_data
# This avoids merging map_data twice and then joining the results

# Pre-join DI with its toggle
DI_lookup = pd.merge(
    DI_data,
    DI_Toggle[['Tier_gap', 'AirCraftType', 'DI Toggle']],
    on=['Tier_gap', 'AirCraftType'],
    how='left'
).rename(columns={'Tier_gap': 'DI Level'})

# Pre-join LI with its toggle
LI_lookup = pd.merge(
    LI_data,
    LI_Toggle[['WU_LI_gap', 'AirCraftType', 'LI Toggle']],
    on=['WU_LI_gap', 'AirCraftType'],
    how='left'
).rename(columns={'WU_LI_gap': 'LI Level'})

# Merge both lookups into one reference table on the same key
combined_lookup = pd.merge(
    DI_lookup,
    LI_lookup,
    on=['AirCraftType', 'corridor', 'DOW', 'TOD'],
    how='outer'  # outer keeps corridors that exist in one but not the other
)

# ✅ FIX 4: Single merge with map_data
final_mapped_data = pd.merge(
    map_data,
    combined_lookup,
    on=['AirCraftType', 'corridor', 'DOW', 'TOD'],
    how='left'
)

# Add weights and Final Toggle
final_mapped_data['WT_DI'] = 0.8
final_mapped_data['WT_LI'] = 0.2
final_mapped_data['Final Toggle'] = (
    final_mapped_data['WT_DI'] * final_mapped_data['DI Toggle'] +
    final_mapped_data['WT_LI'] * final_mapped_data['LI Toggle'])

In [12]:
# ✅ Merge final_mapped_data with the actual cleaned data
final_data = pd.merge(
    data,                  # actual cleaned data
    final_mapped_data[['AirCraftType', 'corridor', 'DOW', 'TOD', 
                        'DI Level', 'LI Level', 'DI Toggle', 'LI Toggle', 
                        'WT_DI', 'WT_LI', 'Final Toggle']],
    on=['AirCraftType', 'corridor', 'DOW', 'TOD'],
    how='left'
)

# ✅ Adding Toggle Delta column
# Formula: flightcost / (1 + final_adjustment_pct/100) * (Final Toggle - final_adjustment_pct/100)
final_data['Toggle Delta'] = (
    final_data['flightcost'] / (1 + final_data['final_adjustment_pct'] / 100)
) * (final_data['Final Toggle'] - final_data['final_adjustment_pct'] / 100)

In [19]:
# ✅ STEP 1: Diagnose — find where duplicates exist
print(f"data rows              : {len(data)}")
print(f"final_mapped_data rows : {len(final_mapped_data)}")

merge_keys = ['AirCraftType', 'corridor', 'DOW', 'TOD']

# Check duplicates in final_mapped_data on merge keys
dupes = final_mapped_data[final_mapped_data.duplicated(subset=merge_keys, keep=False)]
print(f"\nDuplicate rows in final_mapped_data: {len(dupes)}")
print(dupes.sort_values(merge_keys).head(20))

data rows              : 115071
final_mapped_data rows : 115071

Duplicate rows in final_mapped_data: 113657
       AirCraftType          corridor     DOW          TOD DI Level  \
921           Light  Atlanta->Atlanta  Friday  07:00-09:59      NaN   
1715          Light  Atlanta->Atlanta  Friday  07:00-09:59      NaN   
2218          Light  Atlanta->Atlanta  Friday  07:00-09:59      NaN   
4217          Light  Atlanta->Atlanta  Friday  07:00-09:59      NaN   
51387         Light  Atlanta->Atlanta  Friday  07:00-09:59      NaN   
52740         Light  Atlanta->Atlanta  Friday  07:00-09:59      NaN   
53633         Light  Atlanta->Atlanta  Friday  07:00-09:59      NaN   
54239         Light  Atlanta->Atlanta  Friday  07:00-09:59      NaN   
54615         Light  Atlanta->Atlanta  Friday  07:00-09:59      NaN   
54842         Light  Atlanta->Atlanta  Friday  07:00-09:59      NaN   
54948         Light  Atlanta->Atlanta  Friday  07:00-09:59      NaN   
84700         Light  Atlanta->Atlanta  

In [14]:
print(f"Unmatched rows (NaN Toggle Delta): {final_data['Toggle Delta'].isna().sum()}")

Unmatched rows (NaN Toggle Delta): 76476


In [21]:
# ✅ Create a lookup Series/dict from combined_lookup — no merge needed
# Build a lookup index from combined_lookup
combined_lookup_indexed = combined_lookup.set_index(merge_keys)

# ✅ Set index on map_data to join
map_data_indexed = map_data.set_index(merge_keys)

# ✅ Join lookup onto map_data using index (no row explosion possible)
joined = map_data_indexed.join(combined_lookup_indexed, how='left')

# ✅ Add weights and Final Toggle
joined['WT_DI'] = 0.8
joined['WT_LI'] = 0.2
joined['Final Toggle'] = (joined['WT_DI'] * joined['DI Toggle']) + (joined['WT_LI'] * joined['LI Toggle'])

# ✅ Reset index and attach back to original data
data_reset = data.reset_index(drop=True)
joined_reset = joined.reset_index(drop=True)

# ✅ Concat horizontally (no key-based join, purely positional)
final_data = pd.concat([data_reset, joined_reset[['DI Level', 'LI Level', 'DI Toggle', 
                                                    'LI Toggle', 'WT_DI', 'WT_LI', 'Final Toggle']]], axis=1)

# ✅ Validate
print(f"Cleaned data rows     : {len(data)}")
print(f"Final data rows       : {len(final_data)}")
assert len(final_data) == len(data), "⚠️ Row count mismatch!"
print("✅ Row count matches!")

# ✅ Add Toggle Delta
final_data['Toggle Delta'] = (
    final_data['flightcost'] / (1 + final_data['final_adjustment_pct'] / 100)
) * (final_data['Final Toggle'] - final_data['final_adjustment_pct'] / 100)

print(f"\nNaN in Final Toggle : {final_data['Final Toggle'].isna().sum()}")
print(f"NaN in Toggle Delta : {final_data['Toggle Delta'].isna().sum()}")
print(final_data[['AirCraftType', 'corridor', 'DOW', 'TOD', 
                   'Final Toggle', 'Toggle Delta']].head(10))

Cleaned data rows     : 115071
Final data rows       : 115071
✅ Row count matches!

NaN in Final Toggle : 6216
NaN in Toggle Delta : 6415
    AirCraftType                      corridor        DOW          TOD  \
0  Super Midsize  South Florida->New York City   Thursday  10:00-12:59   
1  Super Midsize  New York City->South Florida     Sunday  13:00-15:59   
2  Super Midsize           San Antonio->Dallas     Monday  13:00-15:59   
3          Light      Nashville->South Florida     Friday  16:00-18:59   
4          Light      South Florida->Nashville     Sunday  10:00-12:59   
5          Light               Atlanta->Dallas     Friday  10:00-12:59   
6          Light     Pittsburgh->North Florida    Tuesday  16:00-18:59   
7          Light  North Florida->North Florida     Monday  07:00-09:59   
8          Light  South Florida->New York City  Wednesday  07:00-09:59   
9  Super Midsize       Philadelphia->Charlotte   Thursday  16:00-18:59   

   Final Toggle  Toggle Delta  
0          0.13

In [23]:
final_data.to_excel("Final_Data.xlsx",index=False)

## ------------------------------------------------------------
